# LeetCode 3866 — First Unique Even Element bằng Python

Link đề: https://leetcode.com/problems/first-unique-even-element/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def firstUniqueEven(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Biweekly Contest 178, câu 1, Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng `nums`, tìm số **chẵn** xuất hiện **đúng một lần** và đứng **sớm nhất** (theo vị trí trong mảng); không có thì trả `-1`.
2. Phân biệt hai điều kiện độc lập: "chẵn" (`x % 2 == 0`) và "duy nhất" (tần suất bằng 1), và hiểu **"sớm nhất" nghĩa là nhỏ nhất theo chỉ số, không phải nhỏ nhất theo giá trị**.
3. Biết dùng `Counter`, mảng đếm kích thước 101, `set`, `dict` lưu chỉ số xuất hiện đầu tiên.
4. Viết 5 cách khác nhau và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: trả giá trị nhỏ nhất thay vì đầu tiên, tính cả số lẻ, nhầm "unique" với "khác hàng xóm", trả `0` thay vì `-1`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về số nguyên **chẵn đầu tiên** (theo thứ tự xuất hiện trong mảng) mà **chỉ xuất hiện đúng một lần** trong `nums`. Nếu không có số nào như vậy, trả về `-1`. (Số nguyên chẵn là số chia hết cho 2.)

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,4,2,5,4,6]` | `2` | Các số chẵn xuất hiện đúng một lần là `2` và `6`; `2` đứng trước nên đáp án là `2` (`4` xuất hiện hai lần) |
| 2 | `[4,4]` | `-1` | Số chẵn duy nhất là `4` nhưng nó xuất hiện hai lần |

Constraints: `1 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from collections import Counter
for nums in ([3, 4, 2, 5, 4, 6], [4, 4]):
    cnt = Counter(nums)
    print(nums, "->", [(x, cnt[x]) for x in nums if x % 2 == 0])

[3, 4, 2, 5, 4, 6] -> [(4, 2), (2, 1), (4, 2), (6, 1)]
[4, 4] -> [(4, 2), (4, 2)]


## 3. Kiến thức nền

### 3.1. Số chẵn và toán tử `%`

`x % 2` là phần dư khi chia cho 2: `0` với số chẵn, `1` với số lẻ. Điều kiện chẵn: `x % 2 == 0` (hoặc `x & 1 == 0`).

In [2]:
print([x for x in range(1, 11) if x % 2 == 0])
print(4 % 2, 7 % 2, 6 & 1, 7 & 1)

[2, 4, 6, 8, 10]
0 1 0 1


### 3.2. `Counter` đếm tần suất

`Counter(nums)` cho từ điển `giá_trị -> số_lần`. Truy cập `cnt[x]` với `x` vắng mặt trả `0`.

In [3]:
from collections import Counter
cnt = Counter([3, 4, 2, 5, 4, 6])
print(cnt, cnt[4], cnt[2], cnt[99])

Counter({4: 2, 3: 1, 2: 1, 5: 1, 6: 1}) 2 1 0


### 3.3. `next(generator, default)`

`next(it, default)` lấy phần tử đầu tiên của iterator, nếu cạn thì trả `default`. Kết hợp với generator ta viết được "tìm phần tử đầu tiên thỏa điều kiện hoặc `-1`".

In [4]:
nums = [3, 4, 2, 5, 4, 6]
print(next((x for x in nums if x > 4), -1))
print(next((x for x in nums if x > 100), -1))

5
-1


### 3.4. `list.count(x)`

`lst.count(x)` đếm số lần `x` xuất hiện trong list, mỗi lần gọi là `O(n)`. Tiện cho brute force nhưng chậm nếu gọi lặp lại nhiều lần.

In [5]:
nums = [3, 4, 2, 5, 4, 6]
print(nums.count(4), nums.count(2), nums.count(9))

2 1 0


### 3.5. `dict` lưu chỉ số xuất hiện đầu tiên và `set`

`setdefault(x, i)` chỉ gán `i` nếu `x` chưa có; nhờ vậy ta lưu **chỉ số xuất hiện đầu tiên**. `set` dùng để nhớ "đã thấy" và "đã trùng".

In [6]:
first = {}
for i, x in enumerate([3, 4, 2, 5, 4, 6]):
    first.setdefault(x, i)
print(first)
seen, dup = set(), set()
for x in [3, 4, 2, 5, 4, 6]:
    (dup if x in seen else seen).add(x)
print(seen, dup, seen - dup)

{3: 0, 4: 1, 2: 2, 5: 3, 6: 5}
{2, 3, 4, 5, 6} {4} {2, 3, 5, 6}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Một phần tử đáp án phải thỏa **ba** điều kiện: (1) chẵn; (2) tần suất đúng bằng 1; (3) trong số các phần tử thỏa (1) và (2), đứng **sớm nhất**.

Tính tay `[3, 4, 2, 5, 4, 6]` bằng bảng tần suất:

| giá trị | 3 | 4 | 2 | 5 | 6 |
|---|---|---|---|---|---|
| tần suất | 1 | 2 | 1 | 1 | 1 |
| chẵn? | không | có | có | không | có |
| đủ cả hai? | không | không | **có** | không | **có** |

Hai ứng viên `2` (chỉ số 2) và `6` (chỉ số 5): lấy `2` vì chỉ số nhỏ hơn. Nếu đề hỏi "giá trị nhỏ nhất" thì ví dụ này trùng hợp cũng ra `2`, nhưng với `[6, 2]` đáp án đúng là `6` (đứng đầu), không phải `2`.

Cách nghĩ tự nhiên: đếm tần suất **một lần**, rồi **quét mảng từ trái sang phải** và dừng ở phần tử đầu tiên chẵn và có tần suất 1. Quét theo thứ tự mảng giải quyết yêu cầu "sớm nhất".

In [7]:
def explain(nums):
    cnt = Counter(nums)
    for i, x in enumerate(nums):
        print(f"  i={i} x={x} chẵn={x % 2 == 0} tần suất={cnt[x]}")
        if x % 2 == 0 and cnt[x] == 1:
            return x
    return -1

print(explain([3, 4, 2, 5, 4, 6]))
print(explain([6, 2]))
print(explain([4, 4]))

  i=0 x=3 chẵn=False tần suất=1
  i=1 x=4 chẵn=True tần suất=2
  i=2 x=2 chẵn=True tần suất=1
2
  i=0 x=6 chẵn=True tần suất=1
6
  i=0 x=4 chẵn=True tần suất=2
  i=1 x=4 chẵn=True tần suất=2
-1


## 5. Cách 1 — Brute Force / baseline

Với mỗi vị trí `i` từ trái sang phải: nếu `nums[i]` chẵn **và** `nums.count(nums[i]) == 1` thì trả về luôn. Hết vòng lặp mà không thấy thì `-1`.

Nút thắt: `nums.count` quét cả mảng mỗi lần nên tổng `O(n²)`, và ta đếm lặp lại cùng một giá trị nhiều lần. Một bảng đếm xây một lần giải quyết được.

In [8]:
class SolutionBrute:
    def firstUniqueEven(self, nums):
        for x in nums:
            if x % 2 == 0 and nums.count(x) == 1:
                return x
        return -1

print(SolutionBrute().firstUniqueEven([3, 4, 2, 5, 4, 6]), SolutionBrute().firstUniqueEven([4, 4]))

2 -1


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách làm hai pha rõ ràng:

1. **Pha đếm:** duyệt `nums` một lần, xây bảng tần suất `cnt` (mỗi giá trị chỉ đếm một lần).
2. **Pha chọn:** duyệt `nums` **theo thứ tự gốc**, trả về phần tử đầu tiên thỏa `x % 2 == 0 and cnt[x] == 1`.

**Invariant:** ở pha 2, mọi phần tử đã duyệt trước đó **không** thỏa (hoặc lẻ, hoặc xuất hiện nhiều hơn một lần), nên phần tử đầu tiên thỏa chính là đứng sớm nhất. Việc duyệt theo thứ tự gốc (không duyệt `cnt`, không duyệt `sorted`) bảo đảm tie-break "sớm nhất".

Tối ưu bộ nhớ/hằng số: bảng đếm có thể là mảng 101 phần tử (vì giá trị `<= 100`), hoặc ta chỉ cần đếm các số **chẵn**.

## 7. Các cách tiếp theo

### Cách 2 — `Counter` rồi quét theo thứ tự gốc

Đúng như mục 6, dùng `Counter`.

In [9]:
from collections import Counter

class SolutionCounter:
    def firstUniqueEven(self, nums):
        cnt = Counter(nums)
        for x in nums:
            if x % 2 == 0 and cnt[x] == 1:
                return x
        return -1

print(SolutionCounter().firstUniqueEven([3, 4, 2, 5, 4, 6]), SolutionCounter().firstUniqueEven([4, 4]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(m)` với `m` là số giá trị phân biệt.

### Cách 3 — Mảng đếm kích thước 101

Vì `1 <= nums[i] <= 100`, dùng `freq = [0] * 101`, `freq[x] += 1`; không cần băm.

In [10]:
class SolutionArray:
    def firstUniqueEven(self, nums):
        freq = [0] * 101
        for x in nums:
            freq[x] += 1
        for x in nums:
            if x % 2 == 0 and freq[x] == 1:
                return x
        return -1

print(SolutionArray().firstUniqueEven([3, 4, 2, 5, 4, 6]), SolutionArray().firstUniqueEven([4, 4]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(101)` hằng số.

### Cách 4 — Hai tập `seen` / `dup` (một lượt đếm không cần số)

Duyệt `nums`: nếu `x` đã thấy thì cho vào `dup`, ngược lại cho vào `seen`. Tập "xuất hiện đúng một lần" là `seen - dup`. Rồi quét theo thứ tự gốc.

In [11]:
class SolutionTwoSets:
    def firstUniqueEven(self, nums):
        seen, dup = set(), set()
        for x in nums:
            if x in seen:
                dup.add(x)
            else:
                seen.add(x)
        unique = seen - dup
        for x in nums:
            if x % 2 == 0 and x in unique:
                return x
        return -1

print(SolutionTwoSets().firstUniqueEven([3, 4, 2, 5, 4, 6]), SolutionTwoSets().firstUniqueEven([4, 4]))

2 -1


**Độ phức tạp:** Time `O(n)`, Space `O(m)`.

### Cách 5 — Chỉ số xuất hiện đầu tiên + tần suất, rồi `min` theo chỉ số

Lưu `first[x]` (chỉ số đầu tiên) và `cnt[x]`; sau đó trong các `x` chẵn với `cnt[x] == 1` chọn `x` có `first[x]` nhỏ nhất. Cách này cho thấy rõ "sớm nhất = chỉ số nhỏ nhất" và không cần quét lại mảng.

In [12]:
class SolutionFirstIndex:
    def firstUniqueEven(self, nums):
        first, cnt = {}, {}
        for i, x in enumerate(nums):
            first.setdefault(x, i)
            cnt[x] = cnt.get(x, 0) + 1
        candidates = [x for x in cnt if x % 2 == 0 and cnt[x] == 1]
        if not candidates:
            return -1
        return min(candidates, key=lambda v: first[v])

print(SolutionFirstIndex().firstUniqueEven([3, 4, 2, 5, 4, 6]), SolutionFirstIndex().firstUniqueEven([4, 4]), SolutionFirstIndex().firstUniqueEven([6, 2]))

2 -1 6


**Độ phức tạp:** Time `O(n + m)`, Space `O(m)`.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [3, 4, 2, 5, 4, 6]`. Pha 1: `cnt = {3:1, 4:2, 2:1, 5:1, 6:1}`. Pha 2:

| i | x | chẵn? | `cnt[x] == 1`? | hành động |
|---|---|---|---|---|
| 0 | 3 | không | | bỏ qua |
| 1 | 4 | có | không (2) | bỏ qua |
| 2 | 2 | có | có | **trả về 2** |

In [13]:
def trace_counter(nums):
    cnt = Counter(nums)
    print("pha 1, bảng đếm:", dict(cnt))
    for i, x in enumerate(nums):
        even = x % 2 == 0
        uniq = cnt[x] == 1
        print(f"i={i} x={x}: chẵn={even}, duy nhất={uniq}", end="")
        if even and uniq:
            print("  -> TRẢ VỀ")
            return x
        print()
    print("không có -> -1")
    return -1

assert trace_counter([3, 4, 2, 5, 4, 6]) == 2
assert trace_counter([4, 4]) == -1
assert trace_counter([6, 2]) == 6

pha 1, bảng đếm: {3: 1, 4: 2, 2: 1, 5: 1, 6: 1}
i=0 x=3: chẵn=False, duy nhất=True
i=1 x=4: chẵn=True, duy nhất=False
i=2 x=2: chẵn=True, duy nhất=True  -> TRẢ VỀ
pha 1, bảng đếm: {4: 2}
i=0 x=4: chẵn=True, duy nhất=False
i=1 x=4: chẵn=True, duy nhất=False
không có -> -1
pha 1, bảng đếm: {6: 1, 2: 1}
i=0 x=6: chẵn=True, duy nhất=True  -> TRẢ VỀ


In [14]:
def trace_two_sets(nums):
    seen, dup = set(), set()
    for x in nums:
        (dup if x in seen else seen).add(x)
        print(f"x={x}: seen={sorted(seen)} dup={sorted(dup)}")
    print("duy nhất:", sorted(seen - dup))

trace_two_sets([3, 4, 2, 5, 4, 6])

x=3: seen=[3] dup=[]
x=4: seen=[3, 4] dup=[]
x=2: seen=[2, 3, 4] dup=[]
x=5: seen=[2, 3, 4, 5] dup=[]
x=4: seen=[2, 3, 4, 5] dup=[4]
x=6: seen=[2, 3, 4, 5, 6] dup=[4]
duy nhất: [2, 3, 5, 6]


## 9. Code nộp LeetCode

Dùng Cách 2 (`Counter` + quét theo thứ tự gốc).

In [15]:
from typing import List, Optional
from collections import Counter


class Solution:
    def firstUniqueEven(self, nums: List[int]) -> int:
        cnt = Counter(nums)
        for x in nums:                       # quét theo thứ tự gốc => "sớm nhất"
            if x % 2 == 0 and cnt[x] == 1:
                return x
        return -1


print(Solution().firstUniqueEven([3, 4, 2, 5, 4, 6]), Solution().firstUniqueEven([4, 4]))

2 -1


## 10. Vì sao thuật toán đúng

1. **Bảng đếm chính xác.** `cnt[x]` bằng số lần `x` xuất hiện trong toàn mảng, nên "xuất hiện đúng một lần" tương đương `cnt[x] == 1`.
2. **Điều kiện hợp lệ.** Một phần tử là ứng viên khi `x % 2 == 0` và `cnt[x] == 1`; đây đúng là mô tả của đề.
3. **Tính "sớm nhất".** Ta duyệt theo chỉ số tăng dần và trả về ứng viên đầu tiên gặp. Mọi ứng viên khác đứng sau nó (chỉ số lớn hơn) nên nó là sớm nhất. Vì ứng viên xuất hiện đúng một lần nên "chỉ số xuất hiện" là duy nhất, không mơ hồ.
4. **Không có ứng viên.** Vòng lặp kết thúc mà không trả về -> trả `-1`.
5. Cách 5 chọn `min` theo `first[x]` -> cùng đáp án; Cách 4 thay `Counter` bằng hai tập mô tả cùng tập "xuất hiện đúng một lần".

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `m` là số giá trị phân biệt (`m <= min(n, 100)`).

- Cách 1: `O(n²)` thời gian (mỗi `count` `O(n)`), `O(1)` bộ nhớ.
- Cách 2, 4, 5: `O(n)` thời gian, `O(m)` bộ nhớ (băm).
- Cách 3: `O(n)` thời gian, `O(101)` bộ nhớ.

Best case: phần tử đầu là chẵn và duy nhất -> pha 2 dừng ngay (pha 1 vẫn `O(n)`). Worst case: không có ứng viên -> duyệt hết mảng ở cả hai pha.

## 12. Bộ test

Đối chiếu 5 cách + code nộp với ví dụ đề, biên và 400 input ngẫu nhiên.

In [16]:
import random

solutions = [SolutionBrute(), SolutionCounter(), SolutionArray(), SolutionTwoSets(), SolutionFirstIndex(), Solution()]

fixed = [
    ([3, 4, 2, 5, 4, 6], 2),
    ([4, 4], -1),
    ([2], 2),
    ([1], -1),
    ([1, 3, 5], -1),
    ([6, 2], 6),                    # sớm nhất theo vị trí, không phải nhỏ nhất theo giá trị
    ([2, 3, 2], -1),
    ([100, 1, 1, 100, 98], 98),
    ([8, 8, 8], -1),
    ([5, 10, 5, 10, 12], 12),
]
for nums, expected in fixed:
    for s in solutions:
        assert s.firstUniqueEven(list(nums)) == expected, (type(s).__name__, nums)

random.seed(38660)
for _ in range(400):
    n = random.randint(1, 100)
    hi = random.choice([4, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    outs = [s.firstUniqueEven(list(nums)) for s in solutions]
    assert all(o == outs[0] for o in outs), (nums, outs)
    r = outs[0]
    if r != -1:
        assert r % 2 == 0 and nums.count(r) == 1
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Nếu thêm **ở cuối** một phần tử trùng với đáp án thì đáp án biến mất hoặc đổi sang ứng viên khác (không còn là phần tử cũ). (2) Mảng chỉ gồm số lẻ luôn cho `-1`. (3) Đáp án (nếu có) luôn nằm trong `nums`.

In [17]:
random.seed(3866)
for _ in range(300):
    nums = [random.randint(1, 20) for _ in range(random.randint(1, 40))]
    r = Solution().firstUniqueEven(nums)
    if r != -1:
        assert r in nums
        assert Solution().firstUniqueEven(nums + [r]) != r
    odds = [x for x in nums if x % 2 == 1]
    if odds:
        assert Solution().firstUniqueEven(odds) == -1
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả về số chẵn duy nhất **nhỏ nhất** thay vì **đầu tiên**

Đề yêu cầu "sớm nhất trong mảng". Dùng `min` trên tập ứng viên cho đáp án khác khi giá trị nhỏ đứng sau.

In [18]:
def wrong_min_value(nums):
    cnt = Counter(nums)
    cands = [x for x in cnt if x % 2 == 0 and cnt[x] == 1]
    return min(cands) if cands else -1                          # SAI: nhỏ nhất theo giá trị

print(wrong_min_value([6, 2]), "!=", Solution().firstUniqueEven([6, 2]))
assert wrong_min_value([6, 2]) == 2 and Solution().firstUniqueEven([6, 2]) == 6

2 != 6


### Lỗi 2: quên điều kiện chẵn (trả phần tử duy nhất đầu tiên)

Nếu chỉ kiểm tra `cnt[x] == 1` ta trả luôn số lẻ như `3`.

In [19]:
def wrong_no_even(nums):
    cnt = Counter(nums)
    for x in nums:
        if cnt[x] == 1:                                         # SAI: thiếu x % 2 == 0
            return x
    return -1

print(wrong_no_even([3, 4, 2, 5, 4, 6]), "!=", Solution().firstUniqueEven([3, 4, 2, 5, 4, 6]))
assert wrong_no_even([3, 4, 2, 5, 4, 6]) == 3 and Solution().firstUniqueEven([3, 4, 2, 5, 4, 6]) == 2

3 != 2


### Lỗi 3: nhầm "duy nhất" thành "khác hàng xóm"

Chỉ so với phần tử kề không đủ: `2` ở `[2, 3, 2]` khác hàng xóm của nó nhưng vẫn xuất hiện **hai lần**.

In [20]:
def wrong_neighbors(nums):
    for i, x in enumerate(nums):
        left_ok = i == 0 or nums[i - 1] != x
        right_ok = i == len(nums) - 1 or nums[i + 1] != x
        if x % 2 == 0 and left_ok and right_ok:                 # SAI: không đếm toàn mảng
            return x
    return -1

print(wrong_neighbors([2, 3, 2]), "!=", Solution().firstUniqueEven([2, 3, 2]))
assert wrong_neighbors([2, 3, 2]) == 2 and Solution().firstUniqueEven([2, 3, 2]) == -1

2 != -1


### Lỗi 4: trả về chỉ số thay vì giá trị, hoặc `0` thay vì `-1`

Đề trả về **giá trị** của phần tử. Trả `0` khi không có đáp án cũng sai vì `-1` mới là quy ước.

In [21]:
def wrong_index(nums):
    cnt = Counter(nums)
    for i, x in enumerate(nums):
        if x % 2 == 0 and cnt[x] == 1:
            return i                                            # SAI: trả chỉ số
    return 0                                                    # SAI: phải là -1

print(wrong_index([3, 4, 2, 5, 4, 6]), wrong_index([4, 4]), "!=", Solution().firstUniqueEven([3, 4, 2, 5, 4, 6]), Solution().firstUniqueEven([4, 4]))
assert wrong_index([3, 4, 2, 5, 4, 6]) == 2 and wrong_index([4, 4]) == 0 and Solution().firstUniqueEven([4, 4]) == -1
assert wrong_index([5, 8]) == 1 and Solution().firstUniqueEven([5, 8]) == 8

2 0 != 2 -1


### Lỗi 5: duyệt `Counter` thay vì duyệt `nums`

Duyệt `cnt.items()` cũng theo thứ tự chèn nên đôi khi đúng, nhưng với `set` hoặc khi dùng `sorted`, thứ tự không còn là thứ tự trong mảng. Ví dụ duyệt `sorted(cnt)` trả `2` cho `[6, 2]`.

In [22]:
def wrong_sorted_scan(nums):
    cnt = Counter(nums)
    for x in sorted(cnt):                                       # SAI: thứ tự giá trị, không phải thứ tự mảng
        if x % 2 == 0 and cnt[x] == 1:
            return x
    return -1

print(wrong_sorted_scan([6, 2]), "!=", Solution().firstUniqueEven([6, 2]))
assert wrong_sorted_scan([6, 2]) == 2 and Solution().firstUniqueEven([6, 2]) == 6

2 != 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phần tử **đầu tiên** thỏa điều kiện về **tần suất**" -> đếm một lần rồi quét theo thứ tự gốc. Đây là mẫu của *First Unique Character* (387).
- Hai điều kiện độc lập (chẵn, duy nhất): viết thành `if cond1 and cond2`, đừng gộp bằng suy diễn.
- Khi giá trị nhỏ (`<= 100`) dùng mảng đếm thay cho `Counter` để nhanh và đơn giản.
- Đọc kỹ "đầu tiên/sớm nhất" (theo vị trí) khác với "nhỏ nhất" (theo giá trị).
- Test: mọi số lẻ, một phần tử, tất cả trùng, ứng viên đứng sau số nhỏ hơn.

## 15. Biến thể của bài

1. **First Unique Character in a String** (387): ký tự đầu tiên xuất hiện một lần.
2. **Single Number** (136): phần tử xuất hiện một lần (còn lại hai lần) bằng XOR.
3. **Find Unique Binary String / Unique Number of Occurrences** (1207): làm việc với tần suất.
4. **Find the First Unique Number**-style (LeetCode 1429 Premium): hàng đợi các số duy nhất.
5. Biến thể: số **lẻ** đầu tiên duy nhất; hoặc **số lớn nhất** xuất hiện đúng một lần (LeetCode 1133 Premium).
6. Biến thể: trả về **chỉ số** của phần tử đó; hoặc trả về **tất cả** số chẵn duy nhất theo thứ tự.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | `nums.count(x)` mỗi lần | `O(n²)` | `O(1)` | Oracle |
| 2. `Counter` | đếm rồi quét gốc | `O(n)` | `O(m)` | Nộp bài |
| 3. Mảng đếm | `freq[0..100]` | `O(n)` | `O(101)` | Giá trị nhỏ, nhanh |
| 4. Hai tập | `seen - dup` | `O(n)` | `O(m)` | Khi chỉ cần biết "duy nhất" |
| 5. `first` + `min` | chỉ số đầu tiên + chọn nhỏ nhất | `O(n)` | `O(m)` | Khi muốn thấy rõ "sớm nhất" |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().firstUniqueEven([9, 8, 7, 8, 6])` và `Solution().firstUniqueEven([2, 2, 4, 4])` ra gì?

**Bài 2 (tính tay):** với `nums = [10, 20, 30, 20, 10, 40, 30]`, lập bảng tần suất và chọn đáp án.

**Bài 3 (điền TODO):** hoàn thành `first_unique_odd(nums)`: số **lẻ** đầu tiên xuất hiện đúng một lần, hoặc `-1`.

**Bài 4 (tự viết lại):** viết `all_unique_evens(nums)` trả về **danh sách** các số chẵn xuất hiện đúng một lần theo thứ tự trong mảng.

**Bài 5 (mini challenge):** viết `last_unique_even(nums)` trả về số chẵn duy nhất **cuối cùng** (đứng muộn nhất), hoặc `-1`.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().firstUniqueEven([9, 8, 7, 8, 6]), Solution().firstUniqueEven([2, 2, 4, 4]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def first_unique_odd_todo(nums):
    cnt = Counter(nums)
    # TODO: quét nums, trả về số lẻ đầu tiên có cnt == 1
    return -1

6 -1


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `8` xuất hiện hai lần; `6` xuất hiện một lần. Bài 2: `40` là số duy nhất xuất hiện một lần.

**Mức 2 (vừa):** Bài 3: giống Cách 2 nhưng điều kiện `x % 2 == 1`. Bài 4: duyệt `nums`, lọc `x % 2 == 0 and cnt[x] == 1`. Bài 5: duyệt ngược `reversed(nums)`.

**Mức 3 (gần đáp án):** Bài 1: `6` và `-1`. Bài 2: `40`. Bài 4: `all_unique_evens([3,4,2,5,4,6]) = [2, 6]`. Bài 5: `last_unique_even([3,4,2,5,4,6]) = 6`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().firstUniqueEven([9, 8, 7, 8, 6]) == 6 and Solution().firstUniqueEven([2, 2, 4, 4]) == -1
assert Solution().firstUniqueEven([10, 20, 30, 20, 10, 40, 30]) == 40

# Bài 3
def first_unique_odd(nums):
    cnt = Counter(nums)
    for x in nums:
        if x % 2 == 1 and cnt[x] == 1:
            return x
    return -1
assert first_unique_odd([2, 3, 3, 5, 4]) == 5 and first_unique_odd([1, 1]) == -1 and first_unique_odd([2]) == -1

# Bài 4
def all_unique_evens(nums):
    cnt = Counter(nums)
    return [x for x in nums if x % 2 == 0 and cnt[x] == 1]
assert all_unique_evens([3, 4, 2, 5, 4, 6]) == [2, 6] and all_unique_evens([4, 4]) == []

# Bài 5
def last_unique_even(nums):
    cnt = Counter(nums)
    for x in reversed(nums):
        if x % 2 == 0 and cnt[x] == 1:
            return x
    return -1
assert last_unique_even([3, 4, 2, 5, 4, 6]) == 6 and last_unique_even([4, 4]) == -1 and last_unique_even([6, 2]) == 2
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "phần tử đầu tiên thỏa điều kiện về tần suất":**

```python
cnt = Counter(nums)
for x in nums:                          # theo thứ tự gốc
    if điều_kiện(x) and cnt[x] == 1:    # ở đây: x % 2 == 0
        return x
return -1
```

**Checklist trước khi submit:**

- [ ] Có kiểm tra **cả hai**: `x % 2 == 0` và `cnt[x] == 1`?
- [ ] Quét theo **thứ tự mảng gốc** (không `sorted`, không `set`)?
- [ ] Trả **giá trị** phần tử (không phải chỉ số) và `-1` khi không có?
- [ ] "Duy nhất" đếm trên **toàn mảng**, không chỉ so hàng xóm?
- [ ] Đã thử `[6, 2]`, `[4, 4]`, `[1, 3, 5]`, `[2, 3, 2]`?
- [ ] Chữ ký đúng `firstUniqueEven(self, nums: List[int]) -> int`?